# Bixi Montreal 2024 - Exploratory Data Analysis
Analyzing 13 million bike trips to understand how Montrealers move around the city.

Data source: [BIXI Montreal Open Data](https://bixi.com/en/open-data/)

This notebook is the original exploratory analysis. For the reusable, production-style
pipeline (chunked cleaning + aggregation used by the live dashboard and Power BI report),
see [`src/build_dataset.py`](../src/build_dataset.py).

> To run this notebook, download the 2024 trip data from the link above, unzip it, and
> place `DonneesOuvertes (2).csv` in `data/raw/` at the project root.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/raw/DonneesOuvertes (2).csv')
df['start_time'] = pd.to_datetime(df['STARTTIMEMS'], unit='ms')
df['end_time'] = pd.to_datetime(df['ENDTIMEMS'], unit='ms')
df['duration_min'] = (df['end_time'] - df['start_time']).dt.total_seconds() / 60
df['hour'] = df['start_time'].dt.hour
df['day_of_week'] = df['start_time'].dt.day_name()
df['month'] = df['start_time'].dt.month
df['month_name'] = df['start_time'].dt.month_name()
df_clean = df[(df['duration_min'] >= 1) & (df['duration_min'] <= 180)].copy()
print(f'Total trips: {len(df_clean):,}')

## 1. Trips by Hour of Day

In [ ]:
plt.figure(figsize=(12, 5))
df_clean['hour'].value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.title('Bixi Trips by Hour of Day - Montreal 2024', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('Number of Trips')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../images/trips_by_hour.png', dpi=150)
plt.show()

## 2. Trips by Month

In [ ]:
month_order = df_clean.groupby('month')['month_name'].first()
monthly_counts = df_clean.groupby('month').size()
monthly_counts.index = month_order.values
plt.figure(figsize=(12, 5))
monthly_counts.plot(kind='bar', color='steelblue')
plt.title('Bixi Trips by Month - Montreal 2024', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../images/trips_by_month.png', dpi=150)
plt.show()

## 3. Top 10 Busiest Stations

In [ ]:
top_stations = df_clean['STARTSTATIONNAME'].value_counts().head(10)
plt.figure(figsize=(12, 6))
top_stations.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Busiest Bixi Start Stations - Montreal 2024', fontsize=14)
plt.xlabel('Number of Trips')
plt.ylabel('Station')
plt.tight_layout()
plt.savefig('../images/top_stations.png', dpi=150)
plt.show()

## 4. Trips by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df_clean['day_of_week'].value_counts().reindex(day_order)
plt.figure(figsize=(10, 5))
day_counts.plot(kind='bar', color='steelblue')
plt.title('Bixi Trips by Day of Week - Montreal 2024', fontsize=14)
plt.xlabel('Day')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../images/trips_by_day.png', dpi=150)
plt.show()

## 5. Average Trip Duration by Borough

In [ ]:
borough_duration = df_clean.groupby('STARTSTATIONARRONDISSEMENT')['duration_min'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(12, 6))
borough_duration.sort_values().plot(kind='barh', color='steelblue')
plt.title('Average Trip Duration by Borough - Montreal 2024', fontsize=14)
plt.xlabel('Average Duration (minutes)')
plt.ylabel('Borough')
plt.tight_layout()
plt.savefig('../images/duration_by_borough.png', dpi=150)
plt.show()

## 6. Top 10 Most Popular Routes

In [ ]:
df_clean['route'] = df_clean['STARTSTATIONNAME'] + ' to ' + df_clean['ENDSTATIONNAME']
top_routes = df_clean['route'].value_counts().head(10)
plt.figure(figsize=(12, 6))
top_routes.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Most Popular Bixi Routes - Montreal 2024', fontsize=14)
plt.xlabel('Number of Trips')
plt.ylabel('Route')
plt.tight_layout()
plt.savefig('../images/top_routes.png', dpi=150)
plt.show()